In [2]:
import pandas as pd
import numpy as np
import re
import string
import html

import nltk
from nltk import word_tokenize
from langdetect import detect_langs


seed = 123

In [3]:
# import data and check missing values
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

# rename target col to labels
train = train.rename(columns={"target": "labels"})

# print missing
print("Train missing values:\n", train.isnull().sum())
print("\nTest missing values:\n", test.isnull().sum())

Train missing values:
 id             0
keyword       61
location    2533
text           0
labels         0
dtype: int64

Test missing values:
 id             0
keyword       26
location    1105
text           0
dtype: int64


In [ ]:
# clean text, store cleaned as col, "text_cleaned"
def clean_text(text_in:str):
    out = text_in
    out = re.sub("\n", " ", out) # strip multiline

    out = re.sub("B1G", "BIG", out) # strip multiline

    out = re.sub(r"http[^\s]+", " ", out) # strip http* link
    out = re.sub(r'--&gt;', '', out)
    out = out.replace(r'&lt;', '')
    out = out.replace(r'&gt;', '')
    out = out.replace(r'&amp;', '')
    out = re.sub(r"@\w+", "@USER", out) # @___ -> @USER
    out = re.sub(r"[" + re.escape('!"$%&()*+,-./:;<=>?[\\]^_`{|}~') + "]", " ", out) # most punctuation -> space
    out = re.sub(r"\d+", " ", out) # strip numbers
    out = re.sub(" ' ", " ", out) # most punctuation -> space

    # final space processing
    out = re.sub(r"\s+", " ", out) # multispace to 1 space
    out = re.sub(r"^\s+", "", out) # trim leading space
    out = re.sub(r"\s+$", "", out) # trim trailing space

    return out

train["text_cleaned"] = train["text"].apply(clean_text)
test["text_cleaned"] = test["text"].apply(clean_text)

# print before and after
_train_index = None
_test_index = None

# _train_index = 5803
# _test_index = 2125

_tmp_train = train.sample(1) if _train_index is None else train.loc[[_train_index]]
print("[Train] Row Index: ", _tmp_train.index[0])
print("[Train] Before clean: ", _tmp_train["text"].values[0])
print("[Train] After clean: ", _tmp_train["text_cleaned"].values[0])
print("===========================================================")
_tmp_test = test.sample(1) if _test_index is None else test.loc[[_test_index]]
print("[Test] Row Index: ", _tmp_test.index[0])
print("[Test] Before clean: ", _tmp_test["text"].values[0])
print("[Test] After clean: ", _tmp_test["text_cleaned"].values[0])

# clean tmp var
del _tmp_train, _tmp_test, _train_index, _test_index

[Train] Row Index:  2878
[Train] Before clean:  California meets drought-fueled fire season with extra crews.. Related Articles: http://t.co/rKDzB0TGC3
[Train] After clean:  California meets drought fueled fire season with extra crews Related Articles
[Test] Row Index:  217
[Test] Before clean:  Christian Attacked by Muslims at the Temple Mount after Waving Israeli Flag via Pamela Geller - ... http://t.co/ajMtU7EHy2
[Test] After clean:  Christian Attacked by Muslims at the Temple Mount after Waving Israeli Flag via Pamela Geller
